In [ ]:
import requests
import pandas as pd


# ---------- Function 1: Get latitude and longitude ----------
def get_coordinates(city):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    try:
        response = requests.get(url, params={"name": city, "count": 1})
        response.raise_for_status()

        result = response.json().get("results")

        if not result:
            print(f"City '{city}' not found.")
            return None, None

        latitude = result[0]["latitude"]
        longitude = result[0]["longitude"]

        return latitude, longitude

    except requests.exceptions.RequestException as error:
        print("Error fetching location:", error)
        return None, None


# ---------- Function 2: Get weather forecast ----------
def get_weather(lat, lon):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,rain,windspeed_10m",
        "timezone": "auto"
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()

        return response.json()

    except requests.exceptions.RequestException as error:
        print("Error fetching weather data:", error)
        return None


# ---------- Function 3: Save data to CSV ----------
def save_to_csv(city, weather_data, hours=8):

    hourly = weather_data["hourly"]

    df = pd.DataFrame({
        "Time": hourly["time"][:hours],
        "Temperature (°C)": hourly["temperature_2m"][:hours],
        "Rain (mm)": hourly["rain"][:hours],
        "Windspeed (km/h)": hourly["windspeed_10m"][:hours]
    })

    filename = city.replace(" ", "_") + "_weather.csv"

    df.to_csv(filename, index=False)

    print(f"\nForecast saved to '{filename}'")
    print(df)


# ---------- Main Function ----------
def main():

    while True:

        city = input("Enter city name: ").strip()

        latitude, longitude = get_coordinates(city)

        if latitude is not None and longitude is not None:

            weather = get_weather(latitude, longitude)

            if weather is not None:
                save_to_csv(city, weather)

        choice = input("\nCheck another city? (yes/no): ").strip().lower()

        if choice != "yes":
            print("Program ended.")
            break


# ---------- Program Entry ----------
if __name__ == "__main__":
    main()

Enter city name: Bhopal

Forecast saved to 'Bhopal_weather.csv'
               Time  Temperature (°C)  Rain (mm)  Windspeed (km/h)
0  2026-07-23T00:00              23.9        0.1              13.2
1  2026-07-23T01:00              23.6        0.1              14.7
2  2026-07-23T02:00              23.5        0.2              12.2
3  2026-07-23T03:00              23.3        0.4              10.9
4  2026-07-23T04:00              23.4        1.6              11.3
5  2026-07-23T05:00              23.3        1.0              10.4
6  2026-07-23T06:00              23.4        0.3               8.9
7  2026-07-23T07:00              23.5        1.4               9.6

Check another city? (yes/no): yes
Enter city name: Indore

Forecast saved to 'Indore_weather.csv'
               Time  Temperature (°C)  Rain (mm)  Windspeed (km/h)
0  2026-07-23T00:00              24.1        0.0              11.8
1  2026-07-23T01:00              23.8        0.0              13.0
2  2026-07-23T02:00              